# LakeFlow System Tables — Query API
**Source:** AW_LakeFlow System Tables Dashboard v0.1  
**SDK:** `databricks-sdk` `WorkspaceClient` → `w.statement_execution`  
All queries run against Databricks System Tables via a SQL Warehouse.

In [0]:
import datetime

# ── Configuration ────────────────────────────────────────────────────────────
WAREHOUSE_ID = "2d8e531640ffa469"

# ── Date range widgets ────────────────────────────────────────────────────────
# Defaults: today and 30 days ago — recomputed dynamically on every run.
# Edit the widget values in the UI to override the date range for all queries.
_today         = datetime.date.today()
_one_month_ago = _today - datetime.timedelta(days=30)

dbutils.widgets.text("start_date", str(_one_month_ago), "Start Date (YYYY-MM-DD)")
dbutils.widgets.text("end_date",   str(_today),          "End Date   (YYYY-MM-DD)")

DEFAULT_START_DATE = dbutils.widgets.get("start_date")
DEFAULT_END_DATE   = dbutils.widgets.get("end_date")

print(f"Date range: {DEFAULT_START_DATE}  →  {DEFAULT_END_DATE}")

In [0]:
import time
from typing import Optional
import pandas as pd

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.sql import (
    StatementState,
    Disposition,
    Format,
    StatementParameterListItem,
)

# WorkspaceClient picks up credentials automatically inside Databricks
w = WorkspaceClient()


def execute_query(
    query: str,
    params: Optional[dict] = None,
    warehouse_id: str = WAREHOUSE_ID,
    row_limit: int = 50_000,
    timeout_s: int = 120,
) -> dict:
    """
    Execute a SQL query on a SQL Warehouse using the Databricks SDK.

    Parameters
    ----------
    query        : SQL string. Use :param_name syntax for named parameters.
    params       : Dict of {name: value} — bound to :param_name placeholders.
    warehouse_id : SQL Warehouse ID.
    row_limit    : Max rows returned (default 50 000).
    timeout_s    : Max seconds to wait for completion before raising TimeoutError.

    Returns
    -------
    dict with keys:
      status           — "SUCCEEDED" | "FAILED" | "CANCELED"
      statement_id     — UUID of the executed statement
      schema           — list of {name, type} dicts
      data             — list of row lists  [[col1, col2, ...], ...]
      row_count        — total rows returned
      execution_time_ms— wall-clock ms on the warehouse
      error            — error message string, or None
    """
    named_params = []
    if params:
        for k, v in params.items():
            named_params.append(StatementParameterListItem(name=k, value=str(v)))

    # Submit — wait inline up to 50 s (SDK/API maximum for wait_timeout)
    response = w.statement_execution.execute_statement(
        statement=query,
        warehouse_id=warehouse_id,
        parameters=named_params or None,
        row_limit=row_limit,
        wait_timeout=f"{min(timeout_s, 50)}s",
        disposition=Disposition.INLINE,
        format=Format.JSON_ARRAY,
    )

    # Poll until terminal state if still running after inline wait
    t0 = time.time()
    while response.status.state in (StatementState.PENDING, StatementState.RUNNING):
        if time.time() - t0 > timeout_s:
            raise TimeoutError(
                f"Query {response.statement_id} did not complete within {timeout_s}s"
            )
        time.sleep(2)
        response = w.statement_execution.get_statement(response.statement_id)

    # Collect rows (handle pagination via chunk index)
    rows = []
    if response.result and response.result.data_array:
        rows.extend(response.result.data_array)

    next_chunk_index = (
        response.result.next_chunk_index
        if response.result
        else None
    )
    while next_chunk_index is not None:
        chunk = w.statement_execution.get_statement_result_chunk_n(
            response.statement_id, next_chunk_index
        )
        if chunk.data_array:
            rows.extend(chunk.data_array)
        next_chunk_index = chunk.next_chunk_index

    # Build schema list
    schema = []
    if response.manifest and response.manifest.schema and response.manifest.schema.columns:
        schema = [
            {"name": c.name, "type": c.type_name.value if c.type_name else "STRING"}
            for c in response.manifest.schema.columns
        ]

    return {
        "status":            response.status.state.value,
        "statement_id":      response.statement_id,
        "schema":            schema,
        "data":              rows,
        "row_count":         (response.manifest.total_row_count if response.manifest else len(rows)),
        "execution_time_ms": int((time.time() - t0) * 1000),
        "error":             (response.status.error.message if response.status.error else None),
    }


def to_df(result: dict) -> pd.DataFrame:
    """Convert an execute_query result dict into a Pandas DataFrame."""
    if result["status"] != "SUCCEEDED":
        raise RuntimeError(f"Query failed: {result['error']}")
    cols = [c["name"] for c in result["schema"]]
    return pd.DataFrame(result["data"], columns=cols)


def to_spark_df(result: dict):
    """Convert an execute_query result dict into a Spark DataFrame."""
    return spark.createDataFrame(to_df(result))

## Filter / Selector Queries
These populate dropdowns — workspace IDs and run-as principals.

In [0]:
# ── [SELECT] Workspace IDs ────────────────────────────────────────────
QUERY_WORKSPACE_IDS = """
WITH ws AS (
  SELECT DISTINCT workspace_id AS workspace
  FROM   system.billing.usage
  WHERE  usage_date BETWEEN :param_start_date AND :param_end_date
)
SELECT workspace FROM ws ORDER BY workspace
"""

def get_workspace_ids(start_date=DEFAULT_START_DATE, end_date=DEFAULT_END_DATE):
    return execute_query(QUERY_WORKSPACE_IDS, params={
        "param_start_date": start_date,
        "param_end_date":   end_date,
    })


# ── [SELECT] Run-as principals ────────────────────────────────────
QUERY_RUN_AS = """
SELECT DISTINCT identity_metadata.run_as AS run_as
FROM   system.billing.usage
WHERE  usage_date BETWEEN :param_start_date AND :param_end_date
ORDER BY run_as ASC
"""

def get_run_as_list(start_date=DEFAULT_START_DATE, end_date=DEFAULT_END_DATE):
    return execute_query(QUERY_RUN_AS, params={
        "param_start_date": start_date,
        "param_end_date":   end_date,
    })

## Overview Queries

In [0]:
QUERY_MOST_EXPENSIVE_JOBS = """
WITH billing AS (
  SELECT
    u.workspace_id,
    u.usage_metadata.job_id          AS job_id,
    u.usage_metadata.dlt_pipeline_id AS pipeline_id,
    u.identity_metadata.run_as       AS run_as,
    u.custom_tags,
    SUM(u.usage_quantity * lp.pricing.default) AS total_cost_usd
  FROM system.billing.usage u
  JOIN system.billing.list_prices lp
    ON  lp.sku_name = u.sku_name
    AND lp.price_start_time <= u.usage_date
    AND (lp.price_end_time IS NULL OR lp.price_end_time > u.usage_date)
  WHERE u.usage_date BETWEEN :param_start_date AND :param_end_date
    AND (u.usage_metadata.job_id IS NOT NULL OR u.usage_metadata.dlt_pipeline_id IS NOT NULL)
    AND (:param_workspace = 'ALL' OR u.workspace_id = :param_workspace)
    AND (:param_run_as    = 'ALL' OR u.identity_metadata.run_as = :param_run_as)
    AND (
          :param_type = 'ALL'
          OR (:param_type = 'Job'      AND u.usage_metadata.job_id      IS NOT NULL)
          OR (:param_type = 'Pipeline' AND u.usage_metadata.dlt_pipeline_id IS NOT NULL)
        )
  GROUP BY 1,2,3,4,5
),
enriched AS (
  SELECT
    COALESCE(j.name, p.name,
             CONCAT(CASE WHEN b.job_id IS NOT NULL THEN 'Job' ELSE 'Pipeline' END,
                    ':', COALESCE(b.job_id, b.pipeline_id)))        AS name,
    CASE WHEN b.job_id IS NOT NULL THEN 'Job' ELSE 'Pipeline' END   AS type,
    b.workspace_id,
    b.run_as,
    b.custom_tags,
    b.total_cost_usd
  FROM billing b
  LEFT JOIN system.lakeflow.jobs      j ON j.job_id      = b.job_id      AND j.workspace_id = b.workspace_id
  LEFT JOIN system.lakeflow.pipelines p ON p.pipeline_id = b.pipeline_id AND p.workspace_id = b.workspace_id
)
SELECT name, type, workspace_id, run_as,
       ROUND(SUM(total_cost_usd), 4) AS total_cost_usd
FROM   enriched
GROUP BY 1,2,3,4
ORDER BY total_cost_usd DESC
LIMIT CAST(:param_limit AS INT)
"""

def get_most_expensive_jobs(
    start_date=DEFAULT_START_DATE, end_date=DEFAULT_END_DATE,
    workspace="ALL", run_as="ALL", type_filter="ALL", limit=50
):
    return execute_query(QUERY_MOST_EXPENSIVE_JOBS, params={
        "param_start_date": start_date, "param_end_date": end_date,
        "param_workspace":  workspace,  "param_run_as":   run_as,
        "param_type":       type_filter, "param_limit":   str(limit),
    })

In [0]:
QUERY_MOST_EXPENSIVE_RUNS = """
WITH run_cost AS (
  SELECT
    u.workspace_id,
    u.usage_metadata.job_id                  AS job_id,
    u.usage_metadata.job_run_id              AS run_id,
    u.usage_metadata.dlt_pipeline_id        AS pipeline_id,
    u.usage_metadata.dlt_update_id           AS update_id,
    u.identity_metadata.run_as               AS run_as,
    SUM(u.usage_quantity * lp.pricing.default) AS cost_usd
  FROM system.billing.usage u
  JOIN system.billing.list_prices lp
    ON  lp.sku_name = u.sku_name
    AND lp.price_start_time <= u.usage_date
    AND (lp.price_end_time IS NULL OR lp.price_end_time > u.usage_date)
  WHERE u.usage_date BETWEEN :param_start_date AND :param_end_date
    AND (u.usage_metadata.job_run_id IS NOT NULL OR u.usage_metadata.dlt_update_id IS NOT NULL)
    AND (:param_workspace = 'ALL' OR u.workspace_id = :param_workspace)
    AND (:param_run_as    = 'ALL' OR u.identity_metadata.run_as = :param_run_as)
    AND (
          :param_type = 'ALL'
          OR (:param_type = 'Job'      AND u.usage_metadata.job_id      IS NOT NULL)
          OR (:param_type = 'Pipeline' AND u.usage_metadata.dlt_pipeline_id IS NOT NULL)
        )
  GROUP BY 1,2,3,4,5,6
)
SELECT
  rc.workspace_id,
  COALESCE(j.name, CONCAT('Job:', rc.job_id))           AS job_name,
  COALESCE(p.name, CONCAT('Pipeline:', rc.pipeline_id)) AS pipeline_name,
  CASE WHEN rc.job_id IS NOT NULL THEN 'Job' ELSE 'Pipeline' END AS type,
  COALESCE(CAST(rc.run_id AS STRING), rc.update_id)     AS run_identifier,
  rc.run_as,
  ROUND(rc.cost_usd, 4)                                 AS cost_usd,
  jrt.result_state,
  jrt.run_duration / 1000                               AS duration_seconds,
  jrt.run_start_time
FROM run_cost rc
LEFT JOIN system.lakeflow.jobs             j   ON j.job_id       = rc.job_id   AND j.workspace_id  = rc.workspace_id
LEFT JOIN system.lakeflow.pipelines        p   ON p.pipeline_id  = rc.pipeline_id AND p.workspace_id = rc.workspace_id
LEFT JOIN system.lakeflow.job_run_timeline jrt ON jrt.job_run_id = rc.run_id   AND jrt.workspace_id = rc.workspace_id
ORDER BY cost_usd DESC
LIMIT CAST(:param_limit AS INT)
"""

def get_most_expensive_runs(
    start_date=DEFAULT_START_DATE, end_date=DEFAULT_END_DATE,
    workspace="ALL", run_as="ALL", type_filter="ALL", limit=50
):
    return execute_query(QUERY_MOST_EXPENSIVE_RUNS, params={
        "param_start_date": start_date, "param_end_date": end_date,
        "param_workspace":  workspace,  "param_run_as":   run_as,
        "param_type":       type_filter, "param_limit":   str(limit),
    })

In [0]:
QUERY_RUN_STATUS_DISTRIBUTION = """
SELECT
  DATE_TRUNC(:param_time_key, jrt.run_start_time) AS period,
  jrt.workspace_id,
  jrt.result_state,
  jrt.termination_code,
  COUNT(*) AS run_count
FROM system.lakeflow.job_run_timeline jrt
WHERE CAST(jrt.run_start_time AS DATE) BETWEEN :param_start_date AND :param_end_date
  AND (:param_workspace = 'ALL' OR jrt.workspace_id = :param_workspace)
  AND jrt.run_start_time IS NOT NULL
GROUP BY 1,2,3,4
ORDER BY period DESC, run_count DESC
"""

def get_run_status_distribution(
    start_date=DEFAULT_START_DATE, end_date=DEFAULT_END_DATE,
    workspace="ALL", time_key="Day"
):
    return execute_query(QUERY_RUN_STATUS_DISTRIBUTION, params={
        "param_start_date": start_date, "param_end_date": end_date,
        "param_workspace":  workspace,  "param_time_key": time_key,
    })

In [0]:
QUERY_HIGHEST_USAGE_INCREASE = """
WITH current_period AS (
  SELECT
    u.workspace_id,
    COALESCE(u.usage_metadata.job_id, u.usage_metadata.dlt_pipeline_id, 'other') AS resource_id,
    CASE WHEN u.usage_metadata.job_id      IS NOT NULL THEN 'Job'
         WHEN u.usage_metadata.dlt_pipeline_id IS NOT NULL THEN 'Pipeline'
         ELSE 'Other' END                                                     AS resource_type,
    SUM(u.usage_quantity * lp.pricing.default)                                AS cost_current
  FROM system.billing.usage u
  JOIN system.billing.list_prices lp
    ON  lp.sku_name = u.sku_name
    AND lp.price_start_time <= u.usage_date
    AND (lp.price_end_time IS NULL OR lp.price_end_time > u.usage_date)
  WHERE u.usage_date BETWEEN :param_start_date AND :param_end_date
    AND (:param_workspace = 'ALL' OR u.workspace_id = :param_workspace)
  GROUP BY 1,2,3
),
prior_period AS (
  SELECT
    u.workspace_id,
    COALESCE(u.usage_metadata.job_id, u.usage_metadata.dlt_pipeline_id, 'other') AS resource_id,
    SUM(u.usage_quantity * lp.pricing.default) AS cost_prior
  FROM system.billing.usage u
  JOIN system.billing.list_prices lp
    ON  lp.sku_name = u.sku_name
    AND lp.price_start_time <= u.usage_date
    AND (lp.price_end_time IS NULL OR lp.price_end_time > u.usage_date)
  WHERE u.usage_date BETWEEN
        DATEADD(DAY, -DATEDIFF(DAY, CAST(:param_start_date AS DATE), CAST(:param_end_date AS DATE)) - 1,
                CAST(:param_start_date AS DATE))
    AND DATEADD(DAY, -1, CAST(:param_start_date AS DATE))
    AND (:param_workspace = 'ALL' OR u.workspace_id = :param_workspace)
  GROUP BY 1,2
)
SELECT
  c.workspace_id,
  c.resource_id,
  c.resource_type,
  ROUND(c.cost_current, 2)                              AS cost_current_usd,
  ROUND(COALESCE(p.cost_prior, 0), 2)                   AS cost_prior_usd,
  ROUND(c.cost_current - COALESCE(p.cost_prior, 0), 2)  AS delta_usd,
  ROUND(
    CASE WHEN COALESCE(p.cost_prior, 0) = 0 THEN NULL
         ELSE (c.cost_current - p.cost_prior) / p.cost_prior * 100 END,
    1)                                                   AS pct_change
FROM current_period c
LEFT JOIN prior_period p USING (workspace_id, resource_id)
ORDER BY delta_usd DESC
LIMIT CAST(:param_limit AS INT)
"""

def get_highest_usage_increase(
    start_date=DEFAULT_START_DATE, end_date=DEFAULT_END_DATE,
    workspace="ALL", limit=25
):
    return execute_query(QUERY_HIGHEST_USAGE_INCREASE, params={
        "param_start_date": start_date, "param_end_date": end_date,
        "param_workspace":  workspace,  "param_limit":    str(limit),
    })

In [0]:
QUERY_USAGE_OVER_TIME = """
SELECT
  DATE_TRUNC(:param_time_key, u.usage_date)  AS period,
  u.workspace_id,
  u.sku_name,
  CASE
    WHEN u.billing_origin_product IN ('JOBS', 'WORKFLOWS') THEN 'Serverless'
    WHEN u.usage_metadata.cluster_id IS NOT NULL           THEN 'Non-serverless'
    ELSE 'Other'
  END                                        AS cluster_category,
  ROUND(SUM(u.usage_quantity * lp.pricing.default), 4) AS cost_usd,
  SUM(u.usage_quantity)                      AS dbus
FROM system.billing.usage u
JOIN system.billing.list_prices lp
  ON  lp.sku_name = u.sku_name
  AND lp.price_start_time <= u.usage_date
  AND (lp.price_end_time IS NULL OR lp.price_end_time > u.usage_date)
WHERE u.usage_date BETWEEN :param_start_date AND :param_end_date
  AND (:param_workspace    = 'ALL' OR u.workspace_id = :param_workspace)
  AND (
        :param_cluster_type = '<ALL CLUSTER TYPES>'
        OR (:param_cluster_type = 'Serverless'     AND u.billing_origin_product IN ('JOBS','WORKFLOWS'))
        OR (:param_cluster_type = 'Non-serverless' AND u.usage_metadata.cluster_id IS NOT NULL
                                                   AND u.billing_origin_product NOT IN ('JOBS','WORKFLOWS'))
      )
GROUP BY 1,2,3,4
ORDER BY period
"""

def get_usage_over_time(
    start_date=DEFAULT_START_DATE, end_date=DEFAULT_END_DATE,
    workspace="ALL", time_key="Day", cluster_type="<ALL CLUSTER TYPES>"
):
    return execute_query(QUERY_USAGE_OVER_TIME, params={
        "param_start_date":   start_date,    "param_end_date":     end_date,
        "param_workspace":    workspace,     "param_time_key":     time_key,
        "param_cluster_type": cluster_type,
    })

## Cluster Analysis Queries

In [0]:
QUERY_CLUSTER_UTILIZATION = """
SELECT
  c.workspace_id,
  c.cluster_id,
  c.cluster_name,
  c.cluster_source,
  ROUND(AVG(ce.driver_cpu_percent),    1)    AS avg_driver_cpu_pct,
  ROUND(AVG(ce.worker_cpu_percent),    1)    AS avg_worker_cpu_pct,
  ROUND(AVG(ce.driver_mem_mb)  / 1024, 2)   AS avg_driver_mem_gb,
  ROUND(AVG(ce.worker_avg_mem_mb) / 1024, 2) AS avg_worker_mem_gb,
  ROUND(SUM(ce.uptime_seconds) / 3600, 2)   AS total_machine_hours,
  COUNT(DISTINCT c.cluster_id)               AS clusters_launched
FROM system.compute.clusters c
JOIN system.compute.cluster_events ce ON ce.cluster_id = c.cluster_id
WHERE c.cluster_source IN ('JOB', 'PIPELINE')
  AND (:param_workspace     = 'ALL' OR c.workspace_id = :param_workspace)
  AND (:param_cpu_threshold = 'ALL'
       OR AVG(ce.worker_cpu_percent) <= CAST(:param_cpu_threshold AS DOUBLE))
GROUP BY 1,2,3,4
ORDER BY avg_worker_cpu_pct ASC
"""

def get_cluster_utilization(workspace="ALL", cpu_threshold="ALL"):
    return execute_query(QUERY_CLUSTER_UTILIZATION, params={
        "param_workspace":     workspace,
        "param_cpu_threshold": str(cpu_threshold),
    })


# ── Cluster CPU Utilization Over Time ──────────────────────────────────
QUERY_CLUSTER_CPU_OVER_TIME = """
SELECT
  DATE_TRUNC(:param_time_key, ce.timestamp)  AS period,
  c.workspace_id,
  c.cluster_name,
  ROUND(AVG(ce.worker_cpu_percent), 1)       AS avg_cpu_pct,
  ROUND(AVG(ce.worker_avg_mem_mb) / 1024, 2) AS avg_mem_gb
FROM system.compute.clusters      c
JOIN system.compute.cluster_events ce ON ce.cluster_id = c.cluster_id
WHERE (:param_workspace = 'ALL' OR c.workspace_id = :param_workspace)
GROUP BY 1,2,3
ORDER BY period
"""

def get_cluster_cpu_over_time(workspace="ALL", time_key="Day"):
    return execute_query(QUERY_CLUSTER_CPU_OVER_TIME, params={
        "param_workspace": workspace,
        "param_time_key":  time_key,
    })

In [0]:
# ── Jobs executing on All-Purpose Clusters ──────────────────────────────
QUERY_JOBS_ON_ALL_PURPOSE = """
SELECT
  jrt.workspace_id,
  jrt.job_id,
  j.name                           AS job_name,
  jrt.cluster_id,
  c.cluster_name,
  COUNT(*)                         AS run_count,
  ROUND(AVG(jrt.run_duration) / 1000, 1) AS avg_duration_seconds
FROM system.lakeflow.job_run_timeline jrt
JOIN system.compute.clusters          c  ON c.cluster_id  = jrt.cluster_id AND c.cluster_source = 'API'
LEFT JOIN system.lakeflow.jobs        j  ON j.job_id      = jrt.job_id     AND j.workspace_id   = jrt.workspace_id
WHERE (:param_workspace = 'ALL' OR jrt.workspace_id = :param_workspace)
GROUP BY 1,2,3,4,5
ORDER BY run_count DESC
"""

def get_jobs_on_all_purpose_clusters(workspace="ALL"):
    return execute_query(QUERY_JOBS_ON_ALL_PURPOSE, params={"param_workspace": workspace})


# ── Top Jobs by Potential Savings ─────────────────────────────────────
QUERY_TOP_JOBS_POTENTIAL_SAVINGS = """
WITH job_cost AS (
  SELECT
    u.workspace_id,
    u.usage_metadata.job_id     AS job_id,
    u.usage_metadata.cluster_id AS cluster_id,
    SUM(u.usage_quantity * lp.pricing.default) AS cost_usd
  FROM system.billing.usage u
  JOIN system.billing.list_prices lp
    ON  lp.sku_name = u.sku_name
    AND lp.price_start_time <= u.usage_date
    AND (lp.price_end_time IS NULL OR lp.price_end_time > u.usage_date)
  WHERE u.usage_metadata.job_id IS NOT NULL
    AND u.usage_metadata.cluster_id IS NOT NULL
    AND (:param_workspace = 'ALL' OR u.workspace_id = :param_workspace)
  GROUP BY 1,2,3
),
with_cpu AS (
  SELECT
    jc.workspace_id, jc.job_id, jc.cost_usd,
    AVG(ce.worker_cpu_percent) AS avg_cpu_pct
  FROM job_cost jc
  JOIN system.compute.cluster_events ce ON ce.cluster_id = jc.cluster_id
  GROUP BY 1,2,3
)
SELECT
  wc.workspace_id,
  wc.job_id,
  j.name                                             AS job_name,
  ROUND(wc.avg_cpu_pct, 1)                           AS avg_cpu_pct,
  ROUND(wc.cost_usd, 2)                              AS cost_usd,
  ROUND(wc.cost_usd * (1 - wc.avg_cpu_pct / 100), 2) AS potential_savings_usd
FROM with_cpu wc
LEFT JOIN system.lakeflow.jobs j ON j.job_id = wc.job_id AND j.workspace_id = wc.workspace_id
WHERE wc.avg_cpu_pct <= CAST(:param_cpu_threshold AS DOUBLE)
ORDER BY potential_savings_usd DESC
LIMIT 50
"""

def get_top_jobs_potential_savings(workspace="ALL", cpu_threshold=30):
    return execute_query(QUERY_TOP_JOBS_POTENTIAL_SAVINGS, params={
        "param_workspace":     workspace,
        "param_cpu_threshold": str(cpu_threshold),
    })

## Operational Observability Queries

In [0]:
# ── Highest Failure Jobs ──────────────────────────────────────────────────
QUERY_HIGHEST_FAILURE_JOBS = """
WITH run_costs AS (
  SELECT
    u.workspace_id,
    u.usage_metadata.job_run_id AS run_id,
    SUM(u.usage_quantity * lp.pricing.default) AS cost_usd
  FROM system.billing.usage u
  JOIN system.billing.list_prices lp
    ON  lp.sku_name = u.sku_name
    AND lp.price_start_time <= u.usage_date
    AND (lp.price_end_time IS NULL OR lp.price_end_time > u.usage_date)
  GROUP BY 1,2
),
job_failures AS (
  SELECT
    jrt.workspace_id,
    jrt.job_id,
    j.name                                                         AS job_name,
    COUNT(*)                                                       AS total_runs,
    SUM(CASE WHEN jrt.result_state = 'FAILED' THEN 1 ELSE 0 END)  AS failed_runs,
    SUM(COALESCE(rc.cost_usd, 0))                                  AS wasted_cost_usd
  FROM system.lakeflow.job_run_timeline jrt
  LEFT JOIN system.lakeflow.jobs   j  ON j.job_id  = jrt.job_id  AND j.workspace_id  = jrt.workspace_id
  LEFT JOIN run_costs              rc ON rc.run_id  = jrt.job_run_id AND rc.workspace_id = jrt.workspace_id
  WHERE CAST(jrt.run_start_time AS DATE) BETWEEN :param_start_date AND :param_end_date
    AND (:param_workspace = 'ALL' OR jrt.workspace_id = :param_workspace)
  GROUP BY 1,2,3
)
SELECT *,
  ROUND(failed_runs * 100.0 / NULLIF(total_runs, 0), 1) AS failure_rate_pct
FROM job_failures
WHERE failed_runs > 0
  AND COALESCE(wasted_cost_usd, 0) >= CAST(:param_min_cost AS DOUBLE)
ORDER BY failed_runs DESC, wasted_cost_usd DESC
"""

def get_highest_failure_jobs(
    start_date=DEFAULT_START_DATE, end_date=DEFAULT_END_DATE,
    workspace="ALL", min_cost=0
):
    return execute_query(QUERY_HIGHEST_FAILURE_JOBS, params={
        "param_start_date": start_date, "param_end_date": end_date,
        "param_workspace":  workspace,  "param_min_cost": str(min_cost),
    })


# ── Most Retried Runs ──────────────────────────────────────────────────
QUERY_MOST_RETRIED_RUNS = """
SELECT
  jrt.workspace_id,
  jrt.job_id,
  j.name                                     AS job_name,
  jrt.job_run_id,
  jrt.attempt_number,
  jrt.result_state,
  jrt.run_start_time,
  ROUND(jrt.run_duration / 1000, 1)          AS duration_seconds,
  ROUND(COALESCE(rc.cost_usd, 0), 4)         AS cost_usd
FROM system.lakeflow.job_run_timeline jrt
LEFT JOIN system.lakeflow.jobs j ON j.job_id = jrt.job_id AND j.workspace_id = jrt.workspace_id
LEFT JOIN (
  SELECT workspace_id, usage_metadata.job_run_id AS run_id,
         SUM(usage_quantity * lp.pricing.default)  AS cost_usd
  FROM system.billing.usage
  JOIN system.billing.list_prices lp
    ON  lp.sku_name = system.billing.usage.sku_name
    AND lp.price_start_time <= system.billing.usage.usage_date
    AND (lp.price_end_time IS NULL OR lp.price_end_time > system.billing.usage.usage_date)
  GROUP BY 1,2
) rc ON rc.run_id = jrt.job_run_id AND rc.workspace_id = jrt.workspace_id
WHERE jrt.attempt_number > 0
  AND CAST(jrt.run_start_time AS DATE) BETWEEN :param_start_date AND :param_end_date
  AND (:param_workspace = 'ALL' OR jrt.workspace_id = :param_workspace)
  AND COALESCE(rc.cost_usd, 0) >= CAST(:param_min_cost AS DOUBLE)
ORDER BY jrt.attempt_number DESC, cost_usd DESC
"""

def get_most_retried_runs(
    start_date=DEFAULT_START_DATE, end_date=DEFAULT_END_DATE,
    workspace="ALL", min_cost=0
):
    return execute_query(QUERY_MOST_RETRIED_RUNS, params={
        "param_start_date": start_date, "param_end_date": end_date,
        "param_workspace":  workspace,  "param_min_cost": str(min_cost),
    })

## Health Recommendation Queries

In [0]:
# ── Jobs without Autoscaling ──────────────────────────────────────────────
QUERY_JOBS_WITHOUT_AUTOSCALING = """
SELECT
  j.workspace_id,
  j.job_id,
  j.name        AS job_name,
  j.created_by,
  j.create_time
FROM system.lakeflow.jobs j
WHERE (:param_workspace = 'ALL' OR j.workspace_id = :param_workspace)
  AND NOT EXISTS (
        SELECT 1
        FROM system.compute.clusters c
        WHERE c.cluster_id = j.job_id
          AND c.autoscale IS NOT NULL
      )
ORDER BY j.create_time DESC
"""

def get_jobs_without_autoscaling(workspace="ALL"):
    return execute_query(QUERY_JOBS_WITHOUT_AUTOSCALING, params={"param_workspace": workspace})


# ── Untagged Jobs ─────────────────────────────────────────────────────────
QUERY_UNTAGGED_JOBS = """
SELECT
  workspace_id, job_id, name AS job_name, created_by, create_time
FROM system.lakeflow.jobs
WHERE (tags IS NULL OR SIZE(tags) = 0)
  AND (:param_workspace = 'ALL' OR workspace_id = :param_workspace)
ORDER BY create_time DESC
"""

def get_untagged_jobs(workspace="ALL"):
    return execute_query(QUERY_UNTAGGED_JOBS, params={"param_workspace": workspace})


# ── Jobs on Older DBR Versions ───────────────────────────────────────────
QUERY_JOBS_OLD_DBR = """
SELECT
  j.workspace_id,
  j.job_id,
  j.name       AS job_name,
  j.created_by,
  c.spark_version AS dbr_version,
  j.create_time
FROM system.lakeflow.jobs    j
JOIN system.compute.clusters c ON c.cluster_id = j.job_id
WHERE c.spark_version IS NOT NULL
  AND CAST(SPLIT(c.spark_version, '\\\\.')[0] AS INT) < 14
  AND (:param_workspace = 'ALL' OR j.workspace_id = :param_workspace)
ORDER BY CAST(SPLIT(c.spark_version, '\\\\.')[0] AS INT) ASC
"""

def get_jobs_old_dbr(workspace="ALL"):
    return execute_query(QUERY_JOBS_OLD_DBR, params={"param_workspace": workspace})


# ── Stale Datasets (Written but never read downstream) ──────────────────
QUERY_STALE_DATASETS = """
WITH written AS (
  SELECT DISTINCT
    tl.target_table_full_name  AS table_name,
    tl.workspace_id,
    tl.source_run_as           AS written_by
  FROM system.access.table_lineage tl
  WHERE tl.event_type IN ('WRITE', 'CREATE', 'OVERWRITE')
    AND (:param_workspace = 'ALL' OR tl.workspace_id = :param_workspace)
),
read_tables AS (
  SELECT DISTINCT target_table_full_name AS table_name
  FROM system.access.table_lineage
  WHERE event_type IN ('READ', 'SELECT')
)
SELECT
  w.workspace_id,
  w.table_name,
  w.written_by,
  'No downstream consumers' AS lineage_status
FROM written w
LEFT JOIN read_tables r ON r.table_name = w.table_name
WHERE r.table_name IS NULL
ORDER BY w.table_name
"""

def get_stale_datasets(workspace="ALL"):
    return execute_query(QUERY_STALE_DATASETS, params={"param_workspace": workspace})

## Interactive Search Query

In [0]:
QUERY_INTERACTIVE_SEARCH = """
WITH base AS (
  SELECT
    u.workspace_id,
    COALESCE(u.usage_metadata.job_id, u.usage_metadata.dlt_pipeline_id, 'other') AS resource_id,
    CASE WHEN u.usage_metadata.job_id      IS NOT NULL THEN 'Job'
         WHEN u.usage_metadata.dlt_pipeline_id IS NOT NULL THEN 'Pipeline'
         ELSE 'Other' END                                                     AS resource_type,
    u.usage_metadata.cluster_id                                               AS cluster_id,
    u.identity_metadata.run_as,
    u.usage_date,
    u.custom_tags,
    u.sku_name,
    u.usage_quantity * lp.pricing.default                                     AS cost_usd
  FROM system.billing.usage u
  JOIN system.billing.list_prices lp
    ON  lp.sku_name = u.sku_name
    AND lp.price_start_time <= u.usage_date
    AND (lp.price_end_time IS NULL OR lp.price_end_time > u.usage_date)
  WHERE u.usage_date BETWEEN :param_start_date AND :param_end_date
    AND (u.usage_metadata.job_id IS NOT NULL OR u.usage_metadata.dlt_pipeline_id IS NOT NULL)
    AND (:param_workspace    = 'ALL' OR u.workspace_id = :param_workspace)
    AND (:param_run_as       = 'ALL' OR u.identity_metadata.run_as = :param_run_as)
    AND (
          :param_cluster_type = '<ALL CLUSTER TYPES>'
          OR (:param_cluster_type = 'Serverless'     AND u.billing_origin_product IN ('JOBS','WORKFLOWS'))
          OR (:param_cluster_type = 'Non-serverless' AND u.usage_metadata.cluster_id IS NOT NULL
                                                     AND u.billing_origin_product NOT IN ('JOBS','WORKFLOWS'))
        )
    AND (
          :param_type = 'ALL'
          OR (:param_type = 'Job'      AND u.usage_metadata.job_id      IS NOT NULL)
          OR (:param_type = 'Pipeline' AND u.usage_metadata.dlt_pipeline_id IS NOT NULL)
        )
)
SELECT
  b.workspace_id,
  COALESCE(j.name, p.name, CONCAT(b.resource_type, ':', b.resource_id)) AS name,
  b.resource_type,
  b.resource_id,
  b.run_as,
  b.sku_name,
  b.usage_date,
  ROUND(SUM(b.cost_usd), 4) AS cost_usd
FROM base b
LEFT JOIN system.lakeflow.jobs      j ON j.job_id      = b.resource_id AND j.workspace_id = b.workspace_id
LEFT JOIN system.lakeflow.pipelines p ON p.pipeline_id = b.resource_id AND p.workspace_id = b.workspace_id
GROUP BY 1,2,3,4,5,6,7
ORDER BY cost_usd DESC
LIMIT CAST(:param_limit AS INT)
"""

def interactive_search(
    start_date=DEFAULT_START_DATE, end_date=DEFAULT_END_DATE,
    workspace="ALL", run_as="ALL",
    cluster_type="<ALL CLUSTER TYPES>", type_filter="ALL",
    limit=100
):
    return execute_query(QUERY_INTERACTIVE_SEARCH, params={
        "param_start_date":   start_date,    "param_end_date":     end_date,
        "param_workspace":    workspace,     "param_run_as":       run_as,
        "param_cluster_type": cluster_type,  "param_type":         type_filter,
        "param_limit":        str(limit),
    })

## Query Registry & Batch Executor

In [0]:
import concurrent.futures

# All named query functions in one registry
QUERY_REGISTRY = {
    # Filters
    "workspace_ids":              lambda: get_workspace_ids(),
    "run_as_list":                lambda: get_run_as_list(),
    # Overview
    "most_expensive_jobs":        lambda: get_most_expensive_jobs(),
    "most_expensive_runs":        lambda: get_most_expensive_runs(),
    "run_status_distribution":    lambda: get_run_status_distribution(),
    "highest_usage_increase":     lambda: get_highest_usage_increase(),
    "usage_over_time":            lambda: get_usage_over_time(),
    # Clusters
    "cluster_utilization":        lambda: get_cluster_utilization(),
    "cluster_cpu_over_time":      lambda: get_cluster_cpu_over_time(),
    "jobs_on_all_purpose":        lambda: get_jobs_on_all_purpose_clusters(),
    "top_jobs_potential_savings": lambda: get_top_jobs_potential_savings(),
    # Operational
    "highest_failure_jobs":       lambda: get_highest_failure_jobs(),
    "most_retried_runs":          lambda: get_most_retried_runs(),
    # Health
    "jobs_without_autoscaling":   lambda: get_jobs_without_autoscaling(),
    "untagged_jobs":              lambda: get_untagged_jobs(),
    "jobs_old_dbr":               lambda: get_jobs_old_dbr(),
    "stale_datasets":             lambda: get_stale_datasets(),
    # Search
    "interactive_search":         lambda: interactive_search(),
}


def run_queries(query_names: list = None, max_workers: int = 4) -> dict:
    """
    Run a subset (or all) queries in parallel.
    max_workers bounded to avoid overwhelming the warehouse.
    """
    targets = {k: v for k, v in QUERY_REGISTRY.items()
               if query_names is None or k in query_names}
    results = {}
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as pool:
        futures = {pool.submit(fn): name for name, fn in targets.items()}
        for future in concurrent.futures.as_completed(futures):
            name = futures[future]
            try:
                results[name] = future.result()
                r = results[name]
                print(f"✅  {name:35s}  {r['row_count']:>6} rows  {r['execution_time_ms']:>7} ms")
            except Exception as exc:
                results[name] = {"status": "ERROR", "error": str(exc)}
                print(f"❌  {name:35s}  ERROR: {exc}")
    return results

## Example Usage

In [0]:
# ── Single query example ──────────────────────────────────────────────────
result = get_most_expensive_jobs(
    start_date="2025-01-01",
    end_date="2026-06-26",
    workspace="ALL",
    limit=20
)

print(f"Status : {result['status']}")
print(f"Rows   : {result['row_count']}")
print(f"Time   : {result['execution_time_ms']} ms")
print(f"Stmt ID: {result['statement_id']}")

if result["row_count"] > 0:
    display(to_df(result))
else:
    print("No rows returned for the given filters.")

In [0]:
# ── Batch run a subset ───────────────────────────────────────────────────
overview_results = run_queries(
    query_names=[
        "workspace_ids",
        "most_expensive_jobs",
        "run_status_distribution",
        "highest_usage_increase",
    ],
    max_workers=4,
)

# Inspect one result as a DataFrame
display(to_df(overview_results["most_expensive_jobs"]))